In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_5.py ──
"""
Shared utilities for Exercise 5 — GANs and Generative Models.

Infrastructure only: data loading, visualisation helpers, metric computation,
and kailash-ml engine setup. No domain logic or business scenarios.
"""

import asyncio
import pickle
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import matplotlib.pyplot as plt
from matplotlib.figure import Figure

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


if TYPE_CHECKING:
    from kailash_ml.engines.model_registry import ModelVersion

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
LATENT_DIM = 64
IMG_DIM = 28 * 28
BATCH_SIZE = 128
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
except NameError:
    REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "mnist"
OUTPUT_DIR = Path("outputs") / "ex5_gans"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# Environment and device
# ════════════════════════════════════════════════════════════════════════
def init_environment() -> torch.device:
    """Set up environment, seeds, and return the compute device."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ════════════════════════════════════════════════════════════════════════
# Data loading
# ════════════════════════════════════════════════════════════════════════
def load_mnist(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, DataLoader]:
    """Load full MNIST (60K), scale to [-1, 1] for tanh generators.

    Returns:
        X_real: (60000, 1, 28, 28) tensor on device, range [-1, 1]
        y_real: (60000,) long tensor on device
        real_loader: DataLoader with batch_size=128, shuffle, drop_last
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_real = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_real = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))], dtype=torch.long
    )
    X_real = (X_real * 2.0 - 1.0).to(device)
    y_real = y_real.to(device)

    print(
        f"MNIST: {len(X_real)} digits, shape {tuple(X_real.shape[1:])}, "
        f"pixel range [{X_real.min():.2f}, {X_real.max():.2f}]"
    )
    class_dist = ", ".join(f"{c}={int((y_real == c).sum())}" for c in range(10))
    print(f"  class distribution: {class_dist}")

    real_loader = DataLoader(
        TensorDataset(X_real), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    return X_real, y_real, real_loader


def load_mnist_test(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Load the 10K MNIST test digits — images no generator was trained on.

    Returns:
        X_test: (10000, 1, 28, 28) tensor on device, range [-1, 1]
        y_test: (10000,) long tensor on device
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    test_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    X_test = torch.stack([test_set[i][0] for i in range(len(test_set))])
    y_test = torch.tensor(
        [test_set[i][1] for i in range(len(test_set))], dtype=torch.long
    )
    return (X_test * 2.0 - 1.0).to(device), y_test.to(device)


# ════════════════════════════════════════════════════════════════════════
# Kailash engine setup
# ════════════════════════════════════════════════════════════════════════
def setup_engines() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None]
):
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry
    """

    async def _setup():
        # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
        # and ModelRegistry use incompatible _kml_model_versions schemas.
        # Route them to separate sqlite files until upstream fixes the conflict.
        db = "sqlite:///mlfp05_gans.db"
        registry_db = "sqlite:///mlfp05_gans_registry.db"
        tracker = await ExperimentTracker.create(store_url=db)
        conn = ConnectionManager(registry_db)
        await conn.initialize()
        registry = ModelRegistry(conn)
        return conn, tracker, "m5_gans", registry

    return asyncio.run(_setup())


async def close_engines(conn: ConnectionManager) -> None:
    """Cleanly shut down the connection manager."""
    await conn.close()


# ════════════════════════════════════════════════════════════════════════
# Generator and Discriminator architectures
# ════════════════════════════════════════════════════════════════════════
class Generator(nn.Module):
    """MLP Generator: z -> 784-d -> (1, 28, 28).

    A fully-connected (MLP) generator, not a convolutional DCGAN. It borrows
    the DCGAN training recipe — BatchNorm + LeakyReLU in hidden layers and a
    Tanh output to match the [-1, 1] image scaling. Linear layers that feed
    BatchNorm have no bias: BatchNorm subtracts the batch mean, so such a
    bias would be cancelled and would receive exactly zero gradient (which
    the Prescription Pad would report as a vanishing-gradient layer).
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden, bias=False),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2, bias=False),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    """MLP Discriminator: 28x28 -> scalar logit.

    Dropout prevents D from overfitting to real images (memorising
    instead of learning distributional features).
    """

    def __init__(self, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_DIM, hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# FID feature extractor
# ════════════════════════════════════════════════════════════════════════
class LeNetFeatureExtractor(nn.Module):
    """CNN feature extractor for FID computation.

    Returns 64-dim feature vectors (analogous to InceptionV3 pool3 layer).
    We use a domain-specific extractor because InceptionV3 expects 299x299 RGB;
    for 28x28 grayscale MNIST a trained LeNet gives more meaningful distances.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def train_feature_extractor(
    X_real: torch.Tensor,
    y_real: torch.Tensor,
    device: torch.device,
    epochs: int = 5,
) -> LeNetFeatureExtractor:
    """Train the LeNet feature extractor on MNIST for FID computation.

    Returns the trained extractor in eval mode.
    """
    print("\n== Training feature extractor (for FID + mode coverage) ==")
    extractor = LeNetFeatureExtractor().to(device)
    opt = torch.optim.Adam(extractor.parameters(), lr=1e-3)
    X_01 = (X_real + 1.0) / 2.0  # [0, 1] for classifier

    for epoch in range(epochs):
        losses = []
        for xb, yb in DataLoader(
            TensorDataset(X_01, y_real), batch_size=256, shuffle=True
        ):
            loss = F.cross_entropy(extractor(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            acc = (extractor(X_01[:10000]).argmax(-1) == y_real[:10000]).float().mean()
        print(f"  epoch {epoch+1}/{epochs}  loss={np.mean(losses):.3f}  acc={acc:.3f}")

    extractor.eval()
    return extractor


# ════════════════════════════════════════════════════════════════════════
# FID computation
# ════════════════════════════════════════════════════════════════════════
def compute_fid(
    extractor: LeNetFeatureExtractor,
    real: torch.Tensor,
    generated: torch.Tensor,
) -> float:
    """Frechet distance between real and generated feature distributions.

    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))

    Computed in the 64-d LeNet feature space (not Inception), so values are
    only comparable with other values from the same extractor. Lower = the
    generated distribution is closer to the real one (fidelity AND
    diversity). Uses eigenvalues of Sigma_r @ Sigma_g (no scipy dependency).
    """
    extractor.eval()

    def _extract(images: torch.Tensor) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for i in range(0, len(images), 512):
                feats.append(
                    extractor.extract_features(images[i : i + 512]).cpu().numpy()
                )
        return np.concatenate(feats)

    rf, gf = _extract(real), _extract(generated)
    mu_r, mu_g = rf.mean(0), gf.mean(0)
    sig_r, sig_g = np.cov(rf, rowvar=False), np.cov(gf, rowvar=False)

    diff = mu_r - mu_g
    # Sigma_r @ Sigma_g is a product of two symmetric PSD matrices — it is
    # NOT itself symmetric, so np.linalg.eigh (Hermitian-only) reads the wrong
    # triangle and returns garbage (this produced large-negative FID). Its
    # eigenvalues are nonetheless real and non-negative (it is similar to the
    # symmetric PSD matrix Sigma_r^½ Sigma_g Sigma_r^½), so use the general
    # eigensolver and the identity Tr(sqrt(AB)) = Σ sqrt(λ_i(AB)).
    eigvals = np.linalg.eigvals(sig_r @ sig_g)
    eigvals = np.maximum(np.real(eigvals), 0.0)
    trace_sqrt = float(np.sum(np.sqrt(eigvals)))

    return float(diff @ diff + np.trace(sig_r) + np.trace(sig_g) - 2 * trace_sqrt)


# ════════════════════════════════════════════════════════════════════════
# Mode coverage diagnostic
# ════════════════════════════════════════════════════════════════════════
def mode_coverage(
    G: Generator,
    classifier: LeNetFeatureExtractor,
    device: torch.device,
    n: int = 5000,
) -> tuple[int, dict[int, int], float]:
    """Measure mode coverage: how many of the 10 digit classes the generator produces.

    Returns:
        n_classes: number of unique classes generated
        per_class_counts: dict mapping class -> count
        shannon_entropy: diversity measure (max = log2(10) = 3.32 for uniform)
    """
    G.eval()
    classifier.eval()
    with torch.no_grad():
        fake_01 = (G(torch.randn(n, LATENT_DIM, device=device)) + 1) / 2
        preds = classifier(fake_01).argmax(-1).cpu().numpy()
    unique, counts = np.unique(preds, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    return int(len(unique)), {int(k): int(v) for k, v in zip(unique, counts)}, entropy


# ════════════════════════════════════════════════════════════════════════
# Visualisation helpers
# ════════════════════════════════════════════════════════════════════════
def plot_image_grid(
    images: torch.Tensor,
    nrow: int = 8,
    ncol: int = 8,
    title: str = "Generated Images",
    save_path: str | None = None,
) -> Figure:
    """Plot an 8x8 grid of generated images.

    Args:
        images: (N, 1, 28, 28) tensor in [-1, 1] range
        nrow, ncol: grid dimensions
        title: figure title
        save_path: optional path to save the figure
    """
    n = min(nrow * ncol, len(images))
    fig, axes = plt.subplots(nrow, ncol, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(nrow * ncol):
        ax = axes[i // ncol][i % ncol]
        if i < n:
            img = images[i].detach().cpu().squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] -> [0,1]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_latent_interpolation(
    G: Generator,
    device: torch.device,
    n_steps: int = 10,
    n_rows: int = 5,
    title: str = "Latent Space Interpolation",
    save_path: str | None = None,
) -> Figure:
    """Interpolate between pairs of random latent vectors.

    Shows smooth transitions between generated images — evidence that
    the generator has learned a continuous manifold, not memorised digits.
    """
    G.eval()
    fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 1.2, n_rows * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    with torch.no_grad():
        for row in range(n_rows):
            z1 = torch.randn(1, LATENT_DIM, device=device)
            z2 = torch.randn(1, LATENT_DIM, device=device)
            for col in range(n_steps):
                alpha = col / (n_steps - 1)
                z = (1 - alpha) * z1 + alpha * z2
                img = G(z).squeeze().cpu().numpy()
                img = (img + 1) / 2
                axes[row][col].imshow(img, cmap="gray", vmin=0, vmax=1)
                axes[row][col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_training_progression(
    G: Generator,
    device: torch.device,
    epoch_snapshots: dict[int, dict],
    title: str = "Training Progression",
    save_path: str | None = None,
) -> Figure:
    """Plot generated images at different training epochs.

    Shows how generation quality improves over training — from random
    noise to recognisable digits.

    Args:
        epoch_snapshots: dict of {epoch: state_dict} captured during training
    """
    n_snapshots = len(epoch_snapshots)
    n_samples = 8
    fig, axes = plt.subplots(
        n_snapshots, n_samples, figsize=(n_samples * 1.4, n_snapshots * 1.6)
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    fixed_z = torch.randn(n_samples, LATENT_DIM, device=device)

    for row, (epoch, state_dict) in enumerate(sorted(epoch_snapshots.items())):
        G.load_state_dict(state_dict)
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_z)
        for col in range(n_samples):
            ax = axes[row][col] if n_snapshots > 1 else axes[col]
            img = imgs[col].squeeze().cpu().numpy()
            img = (img + 1) / 2
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_loss_curves(
    g_losses: list[float],
    d_losses: list[float],
    title: str = "Training Dynamics",
    g_label: str = "Generator",
    d_label: str = "Discriminator",
    save_path: str | None = None,
) -> Figure:
    """Plot G vs D loss curves across epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(g_losses) + 1)

    # Individual losses
    ax1.plot(epochs, g_losses, "b-", linewidth=2, label=g_label)
    ax1.plot(epochs, d_losses, "r-", linewidth=2, label=d_label)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("G vs D Loss", fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # G/D ratio — healthy GAN has ratio near 1
    ratio = [g / (d + 1e-8) for g, d in zip(g_losses, d_losses)]
    ax2.plot(epochs, ratio, "g-", linewidth=2)
    ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Balanced (1.0)")
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("G/D Loss Ratio", fontsize=12)
    ax2.set_title("Training Balance", fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


# ════════════════════════════════════════════════════════════════════════
# Model registration helper
# ════════════════════════════════════════════════════════════════════════
def register_generator(
    registry: ModelRegistry | None,
    name: str,
    model: Generator,
    fid: float,
    coverage: int,
    entropy: float,
) -> "ModelVersion | None":
    """Register a trained generator in ModelRegistry with quality metrics."""
    if registry is None:
        print(f"  ModelRegistry not available — skipping {name}")
        return None

    async def _register():
        ver = await registry.register_model(
            name=f"m5_{name}",
            artifact=pickle.dumps(model.state_dict()),
            metrics=[
                MetricSpec(name="fid_score", value=fid),
                MetricSpec(name="mode_coverage", value=float(coverage)),
                MetricSpec(name="class_entropy", value=entropy),
            ],
        )
        print(
            f"  Registered {name}: v={ver.version}, FID={fid:.2f}, "
            f"coverage={coverage}/10"
        )
        return ver

    return asyncio.run(_register())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 5.1: Vanilla GAN (The Minimax Game)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - The adversarial minimax game: Generator vs Discriminator
#   - Why GANs work (Nash equilibrium intuition for professionals)
#   - Build and train an MLP-based GAN on full MNIST (60K images)
#   - Diagnose training dynamics: when is D "winning" vs healthy balance
#   - Visualise generated digits, training progression, and loss dynamics
#   - Apply synthetic data generation to a Singapore insurer's data
#     scarcity problem — and learn why synthetic data is NOT private
#     by default
#
# PREREQUISITES: M5/ex_1 (autoencoders — generative model foundations)
# ESTIMATED TIME: ~45 min
# DATASET: MNIST — 60,000 real 28x28 grayscale digits
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt



## PHASE 1 — THEORY: The Minimax Game


In [ ]:
# Imagine a counterfeiter (Generator) trying to produce fake banknotes
# that fool a detective (Discriminator). The counterfeiter never sees
# real banknotes — they only learn from the detective's feedback:
# "this one looks fake because the watermark is wrong."
#
# Over time:
#   - The counterfeiter improves their forgeries
#   - The detective gets better at spotting fakes
#   - Eventually they reach a standoff (Nash equilibrium) where the
#     detective can't tell real from fake — that's a trained GAN.
#
# Mathematically, this is a minimax game:
#   min_G max_D [ E[log D(x)] + E[log(1 - D(G(z)))] ]
#
# D wants to maximise: score real images high, fake images low
# G wants to minimise: make D score fake images high
#
# When D is perfectly confused (outputs 0.5 for everything),
# D_loss converges to ln(4) ≈ 1.386.
#
# MODE COLLAPSE: The biggest GAN failure mode. The counterfeiter finds
# ONE type of banknote that fools the detective and keeps making only
# that one. In MNIST terms: the generator only produces 1s because
# they're the simplest digit. We'll measure this with mode coverage.



GAN = Generator + Discriminator in a zero-sum game.

  Generator (counterfeiter):
    - Input: random noise z ~ N(0, 1)
    - Output: fake image that should look real
    - Goal: fool the discriminator

  Discriminator (detective):
    - Input: real OR fake image
    - Output: probability that the image is real
    - Goal: correctly classify real vs fake

  Training alternates:
    1. Train D on a batch of real + fake images
    2. Train G to make D output "real" for fake images

  Nash equilibrium: D outputs 0.5 for everything (can't tell the difference)

  KEY RISK — Mode collapse:
    G discovers one "easy" output (e.g., only digit 1) and stops exploring.
    We detect this by checking whether all 10 digit classes are generated.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: The Adversarial Minimax Game")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: Generator + Discriminator


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: Generator and Discriminator Networks")
print("=" * 70)

device = init_environment()
X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()

# Verify architectures
G_test = Generator().to(device)
D_test = Discriminator().to(device)
z_test = torch.randn(4, LATENT_DIM, device=device)

print(f"\n  Generator: {sum(p.numel() for p in G_test.parameters()):,} parameters")
print(f"    Input:  z ~ N(0, 1), dim={LATENT_DIM}")
print(f"    Output: {tuple(G_test(z_test).shape)} image (28x28)")
print(f"\n  Discriminator: {sum(p.numel() for p in D_test.parameters()):,} parameters")
print(f"    Input:  28x28 image")
print(f"    Output: {tuple(D_test(G_test(z_test)).shape)} scalar logit")



### Checkpoint 1


In [ ]:
assert G_test(z_test).shape == (4, 1, 28, 28), "Generator output shape wrong"
assert D_test(G_test(z_test)).shape == (4, 1), "Discriminator output shape wrong"
del G_test, D_test, z_test
print("\n--- Checkpoint 1 passed --- G and D architectures verified\n")



## PHASE 3 — TRAIN: Vanilla GAN with Binary Cross-Entropy


Train a vanilla GAN with BCE loss, logging to ExperimentTracker.


In [ ]:
# L_D = -E[log D(x)] - E[log(1 - D(G(z)))]    (discriminator loss)
# L_G = -E[log D(G(z))]                         (generator loss — non-saturating)
#
# Why not the textbook minimax G loss, min E[log(1 - D(G(z)))]? Early in
# training D rejects fakes confidently (D(G(z)) ≈ 0), and log(1 - D) is
# flat there — its gradient SATURATES and G learns almost nothing. The
# non-saturating version maximises log D(G(z)) instead: same fixed point,
# but its gradient is LARGEST exactly when D rejects the fakes. In BCE
# terms it is simply "score the fakes against the REAL label".
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Vanilla GAN (BCEWithLogitsLoss)")
print("=" * 70)

EPOCHS = 15
LR = 2e-4


async def train_vanilla_gan(epochs: int = EPOCHS, lr: float = LR):
    G = Generator().to(device)
    D = Discriminator().to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    # TODO: Define BCE loss for adversarial training
    # Hint: D returns a raw logit (no sigmoid) — pick the torch.nn BCE
    #       loss that applies the sigmoid internally (numerically stable).
    bce = ____
    g_losses, d_losses = [], []
    epoch_snapshots = {}

    # Capture initial state (random noise output)
    epoch_snapshots[0] = copy.deepcopy(G.state_dict())

    async with tracker.track(experiment=exp_name, run_name="vanilla_gan") as run:
        await run.log_params(
            {
                "architecture": "Vanilla_GAN_MLP",
                "latent_dim": str(LATENT_DIM),
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": "128",
                "loss_type": "BCEWithLogitsLoss",
                "optimizer": "Adam(0.5,0.999)",
            }
        )

        for epoch in range(epochs):
            eg, ed = [], []
            for (real_batch,) in real_loader:
                bs = real_batch.size(0)

                # ── Train Discriminator ──────────────────────────────
                # D sees real images (label=1) and fake images (label=0)
                z = torch.randn(bs, LATENT_DIM, device=device)
                fake = G(z).detach()
                # TODO: D loss = BCE on real (target=1) + BCE on fake (target=0)
                # Hint: sum two bce terms, one per image source. Targets
                #       must match D's (bs, 1) output and live on `device`.
                loss_d = ____
                opt_d.zero_grad()
                loss_d.backward()
                opt_d.step()

                # ── Train Generator ──────────────────────────────────
                # G wants D to classify its fakes as real (target=1)
                z = torch.randn(bs, LATENT_DIM, device=device)
                # TODO: G loss = BCE on D(G(z)) with target=1 (fool the discriminator)
                # Hint: this is the non-saturating loss from the Phase 3 note —
                #       fresh fakes (NOT detached: G needs the gradient), scored
                #       against the label D gives a real image.
                loss_g = ____
                opt_g.zero_grad()
                loss_g.backward()
                opt_g.step()

                eg.append(loss_g.item())
                ed.append(loss_d.item())

            avg_g, avg_d = float(np.mean(eg)), float(np.mean(ed))
            g_losses.append(avg_g)
            d_losses.append(avg_d)
            await run.log_metrics({"g_loss": avg_g, "d_loss": avg_d}, step=epoch + 1)
            print(
                f"  [Vanilla GAN] epoch {epoch+1:2d}/{epochs}  "
                f"D={avg_d:.3f}  G={avg_g:.3f}"
            )

            # Capture snapshots for progression visualisation
            if (epoch + 1) in {1, 5, 10, 15}:
                epoch_snapshots[epoch + 1] = copy.deepcopy(G.state_dict())

        await run.log_metrics(
            {"final_g_loss": g_losses[-1], "final_d_loss": d_losses[-1]}
        )

    return G, D, g_losses, d_losses, epoch_snapshots


print("\n  Training vanilla GAN on full MNIST (60K images)...")
G_gan, D_gan, gan_g_losses, gan_d_losses, gan_snapshots = asyncio.run(
    train_vanilla_gan()
)



## DIAGNOSTIC CHECKPOINT — Vanilla GAN (track G + D separately)


In [ ]:
# GANs have TWO networks training in an adversarial loop. We run the
# Prescription Pad on BOTH the Generator and the Discriminator so we
# can see which side is "winning" and which side is starving for
# signal. Each loss closure re-runs the REAL objective from training
# (no weights are updated), and `train_losses` replays the per-epoch
# losses captured above.
from kailash_ml.diagnostics import run_diagnostic_checkpoint

diag_bce = nn.BCEWithLogitsLoss()


def _g_loss(m, batch):
    # Generator objective used in training: non-saturating BCE, with the
    # trained discriminator D_gan scoring fresh fakes against "real".
    bs = batch[0].size(0)
    z = torch.randn(bs, LATENT_DIM, device=device)
    return diag_bce(D_gan(m(z)), torch.ones(bs, 1, device=device))


def _d_loss(m, batch):
    # Discriminator objective used in training: real -> 1, fake -> 0.
    bs = batch[0].size(0)
    with torch.no_grad():
        fake = G_gan(torch.randn(bs, LATENT_DIM, device=device))
    return diag_bce(m(batch[0]), torch.ones(bs, 1, device=device)) + diag_bce(
        m(fake), torch.zeros(bs, 1, device=device)
    )


g_diag, g_findings = run_diagnostic_checkpoint(
    G_gan,
    real_loader,
    _g_loss,
    title="Vanilla GAN — Generator",
    n_batches=6,
    train_losses=gan_g_losses,
    show=False,
)
print_prescription_pad(g_findings, "Vanilla GAN — Generator")

d_diag, d_findings = run_diagnostic_checkpoint(
    D_gan,
    real_loader,
    _d_loss,
    title="Vanilla GAN — Discriminator",
    n_batches=6,
    train_losses=gan_d_losses,
    show=False,
)
print_prescription_pad(d_findings, "Vanilla GAN — Discriminator")

# HOW TO READ YOUR TWO PRESCRIPTION PADS (your readings depend on your run):
#
#  GRADIENT FLOW — compare G's pad with D's. If D's gradients are healthy
#     but G's are tiny (vanishing), D is dominating: G is being told
#     "everything you make is fake" without being told how to improve.
#     >> Try: reduce D's learning rate relative to G's, add label
#        smoothing (real target 0.9 instead of 1.0), or switch to
#        WGAN-GP (ex_5/02).
#
#  DEAD NEURONS / SATURATION — the generator ends in Tanh. A high
#     saturated fraction there means many pixels are pinned at pure
#     black/white. Combined with a gallery full of near-identical digits,
#     that is the signature of MODE COLLAPSE (count the distinct digits
#     in the 4A gallery — a healthy run shows all ten).
#
#  LOSS TREND — GAN losses are NOT supposed to fall monotonically. G and
#     D trade wins, so the curves oscillate, and G's loss often RISES as
#     D gets stronger. A WARNING here is not automatically a failure.
#     The failure signature is D's loss flat-lining near 0 (D has won
#     permanently); D's loss hovering near ln(4) ≈ 1.386 means D cannot
#     tell real from fake — the equilibrium you want.
#
#  CONNECT TO ex_5/02: run WGAN-GP and compare its pads with these. The
#  claim to test is "the Wasserstein critic gives G a more informative
#  gradient" — check whether your readings actually support it.



### Checkpoint 2


In [ ]:
assert len(gan_g_losses) == EPOCHS, f"Expected {EPOCHS} epochs, got {len(gan_g_losses)}"
# INTERPRETATION: In a healthy GAN, D loss hovers around ln(4) ~ 1.386,
# meaning D is about 50% accurate (can't tell real from fake). If D loss
# drops toward 0, D has "won" — it separates real from fake perfectly —
# and the feedback G receives stops being informative ("everything you
# make is fake"), so training stalls or collapses.
print("\n--- Checkpoint 2 passed --- vanilla GAN trained\n")



## PHASE 4 — VISUALISE: Generated Gallery, Loss Curves, Progression


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: What Has the Generator Learned?")
print("=" * 70)

# 4A: Generated image gallery (8x8 grid)
print("\n  4A: Gallery of 64 generated digits")
G_gan.eval()
with torch.no_grad():
    z_gallery = torch.randn(64, LATENT_DIM, device=device)
    gallery_images = G_gan(z_gallery)

fig_gallery = plot_image_grid(
    gallery_images,
    title="Vanilla GAN — Generated Digits (8x8 Grid)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png"),
)
plt.show()

# 4B: Training progression (epoch 0 → 1 → 5 → 10 → 15)
print("\n  4B: Training progression — from noise to digits")
G_progression = Generator().to(device)
fig_progression = plot_training_progression(
    G_progression,
    device,
    gan_snapshots,
    title="Vanilla GAN — Training Progression (Epoch 0 → 15)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png"),
)
plt.show()
# Reload the final state after progression visualisation
G_gan.load_state_dict(gan_snapshots[max(gan_snapshots.keys())])
G_gan.eval()

# 4C: G vs D loss dynamics
print("\n  4C: Generator vs Discriminator loss curves")
fig_losses = plot_loss_curves(
    gan_g_losses,
    gan_d_losses,
    title="Vanilla GAN — Training Dynamics",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_losses.png"),
)
plt.show()

# 4D: Latent space interpolation
print("\n  4D: Latent space interpolation — smooth transitions between digits")
fig_interp = plot_latent_interpolation(
    G_gan,
    device,
    title="Vanilla GAN — Latent Interpolation",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_interpolation.png"),
)
plt.show()



### Checkpoint 3


In [ ]:
import os

assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png")
), "Gallery image should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png")
), "Progression image should exist"
# INTERPRETATION: The gallery shows whether the generator produces
# diverse, recognisable digits. The progression shows HOW it learns:
# epoch 0 is pure noise, early epochs show blurry shapes, later epochs
# show distinct digits. If all generated images look the same, that's
# mode collapse — the generator found one "easy" output.
print("\n--- Checkpoint 3 passed --- vanilla GAN visualisations complete\n")

# Also save training curves with ModelVisualizer (HTML interactive)
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()
fig_html = viz.training_history(
    metrics={"GAN G loss": gan_g_losses, "GAN D loss": gan_d_losses},
    x_label="Epoch",
    y_label="Loss",
)
fig_html.write_html(str(OUTPUT_DIR / "ex_5_01_vanilla_training.html"))
print("  Interactive training curves saved to ex_5_01_vanilla_training.html")



## PHASE 5 — APPLY: Synthetic Data for a Singapore Insurer


BUSINESS SCENARIO (illustrative):
  You are a data scientist at a Singapore life insurer. Your team wants
  a fraud-detection model but has only 2,000 labelled claim records —
  too few for robust ML — and every new data-access request goes
  through a lengthy Personal Data Protection Act (PDPA) review.

  IDEA: Train a generative model on the real records and sample
  synthetic records with the same statistical properties, to
  supplement the real data while you develop the model.

  WHAT SYNTHETIC DATA DOES NOT GIVE YOU FOR FREE:
  - Privacy. A GAN is trained on real records and can memorise and
    regenerate them. Synthetic data is NOT automatically non-personal
    data: it needs formal guarantees (e.g. differentially-private
    training) plus memorisation / membership-inference testing, and
    the PDPA call belongs to your data-protection officer.
  - New information. Samples from a model fitted on 2,000 records
    cannot carry more signal than those records. Whether they improve
    the fraud model is an empirical question you answer on a held-out
    set of REAL claims.

  WHAT WE CAN CHECK HERE: whether synthetic records match real ones
  statistically.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Data for Insurance (Data Scarcity)")
print("=" * 70)
print(
)

# Proxy for the insurance scenario, using MNIST:
# Real policyholder "profiles" = real MNIST digits (limited sample)
# Synthetic profiles = GAN-generated digits
# Caveat: G_gan was trained on all 60K digits, not on the 2,000-record
# sample, so this comparison is more favourable than the real scenario.

print("\n  Simulating the insurance data scenario with MNIST as proxy...")

# Step 1: Take a small "real" sample (simulating limited insurance data)
rng = np.random.default_rng(42)
small_sample_idx = rng.choice(len(X_real), 2000, replace=False)
X_small_real = X_real[small_sample_idx]
y_small_real = y_real[small_sample_idx]

print(f"  'Real' policyholder records: {len(X_small_real)}")

# Step 2: Generate synthetic data to supplement
# TODO: Generate 18,000 synthetic records using the trained generator
# Hint: sample latent vectors from the generator's prior, then decode them.
G_gan.eval()
with torch.no_grad():
    z_synthetic = ____  # TODO: 18,000 latent vectors (Checkpoint 4 checks the count)
    X_synthetic = ____  # TODO: decode them with the trained generator

print(f"  Synthetic records generated: {len(X_synthetic)}")
print(f"  Combined dataset: {len(X_small_real) + len(X_synthetic)} records")

# Step 3: Compare real vs synthetic pixel distributions
# TODO: Flatten both tensors for distribution comparison
# Hint: every pixel of every record in ONE 1-D NumPy array (tensors live
#       on `device`, so move them to the CPU first).
X_real_flat = ____
X_synth_flat = ____

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Real vs Synthetic Distribution Comparison\n"
    "(Insurance Policyholder Profile Proxy)",
    fontsize=14,
    fontweight="bold",
)

# TODO: Plot overlapping histograms for feature distribution
# Hint: one Axes.hist call per source on axes[0]. The sources differ in
#       size (2K vs 18K), so normalise to a density; keep them
#       semi-transparent, and label them for the legend below.
____
____
axes[0].set_xlabel("Feature Value", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("Feature Distribution Overlap", fontsize=13)
axes[0].legend(fontsize=11)

# TODO: Plot mean feature values per sample
# Hint: reshape each tensor to (n_records, 784), average over the pixel
#       axis, and convert to NumPy — one number per record.
real_means = ____
synth_means = ____
axes[1].hist(real_means, bins=30, alpha=0.6, label="Real", density=True)
axes[1].hist(synth_means, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[1].set_xlabel("Mean Feature Value per Record", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("Record-Level Distribution", fontsize=13)
axes[1].legend(fontsize=11)

# TODO: Plot variance per sample
# Hint: same reshape as the means, but measure the spread over pixels.
real_vars = ____
synth_vars = ____
axes[2].hist(real_vars, bins=30, alpha=0.6, label="Real", density=True)
axes[2].hist(synth_vars, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[2].set_xlabel("Feature Variance per Record", fontsize=12)
axes[2].set_ylabel("Density", fontsize=12)
axes[2].set_title("Record Diversity", fontsize=13)
axes[2].legend(fontsize=11)

plt.tight_layout()
fig.savefig(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Distribution comparison saved")

# Step 4: Stakeholder-ready summary
print("\n  ┌────────────────────────────────────────────────────────────┐")
print("  │  STAKEHOLDER SUMMARY: Synthetic Data Quality Assessment   │")
print("  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Real records available:       {len(X_small_real):>8,}                  │")
print(f"  │  Synthetic records generated:  {len(X_synthetic):>8,}                  │")
print(
    f"  │  Combined training set:        {len(X_small_real)+len(X_synthetic):>8,}                  │"
)
print(
    f"  │  Data augmentation factor:     {(len(X_small_real)+len(X_synthetic))/len(X_small_real):.1f}x                       │"
)
print("  │                                                            │")
real_mean_val = float(np.mean(real_means))
synth_mean_val = float(np.mean(synth_means))
mean_diff_pct = abs(real_mean_val - synth_mean_val) / (abs(real_mean_val) + 1e-8) * 100
print(f"  │  Mean feature difference:      {mean_diff_pct:>7.1f}%                  │")
print("  │  Privacy / PDPA:               NOT ASSESSED (needs a      │")
print("  │                                memorisation test + DPO)   │")
stats_status = "STATISTICALLY CLOSE" if mean_diff_pct < 5.0 else "DISTRIBUTION GAP"
print(f"  │  Statistical match (<5% diff): {stats_status:<27} │")
print("  │  Next step: does it help a model scored on REAL hold-out?  │")
print("  └────────────────────────────────────────────────────────────┘")



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png")
), "Distribution comparison should exist"
assert len(X_synthetic) == 18000, "Should generate 18K synthetic records"
print("\n--- Checkpoint 4 passed --- insurance application complete\n")



## Cleanup


In [ ]:
await close_engines(conn)



## REFLECTION


VANILLA GAN FUNDAMENTALS:
  [x] The minimax game: Generator (counterfeiter) vs Discriminator (detective)
  [x] Binary cross-entropy loss for adversarial training
  [x] Nash equilibrium: D loss converging to ln(4) ~ 1.386
  [x] Mode collapse risk: generator producing only "easy" outputs

  VISUAL INTUITION:
  [x] Generated digit gallery (8x8 grid) — can you read the digits?
  [x] Training progression: noise → blurry shapes → recognisable digits
  [x] G vs D loss curves — healthy balance vs D domination
  [x] Latent interpolation — smooth transitions prove learned manifold

  REAL-WORLD APPLICATION:
  [x] Synthetic data to supplement scarce training records
  [x] Statistical validation: real vs synthetic distribution comparison
  [x] Why synthetic data is not private by default, and what must be
      tested before anyone treats it as non-personal data

  KEY INSIGHT:
  Vanilla GANs work but are UNSTABLE. The original minimax G loss,
  log(1 - D(G(z))), saturates when D confidently rejects fakes; that is
  why we trained G with the non-saturating loss -log D(G(z)). But the
  game still minimises the JS divergence, and when the real and
  generated distributions barely overlap JS is stuck at its maximum
  (log 2) — it says "different" without saying "how far". D can win,
  G chases a moving target, and modes collapse.

  Next: Exercise 5.2 — WGAN-GP solves this instability with
  Wasserstein distance (smooth gradients even when distributions
  don't overlap) and gradient penalty (replaces weight clipping).


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

